### Random forest MTL na deskryptorach - zestaw 1 - Absorpcja**

Wykorzystana reprezentacja: **ECFP4**

Lista endpointów:


1. Caco-2 (Wang)
2. Lipophilicity (AstraZeneca)
3. Solubility (AqSolDB)
4. HIA (Hou)
5. AMES Mutagenicity

Wyniki dla STL:

In [1]:
!pip install rdkit
!pip install pandas numpy scikit-learn -U
!pip install pytdc --no-dependencies
!pip install fuzzywuzzy

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
data_folder = "/content/drive/MyDrive/mldd_data" #folder z zapisanymi zestawami danych aby umożliwić porównanie danych na dokładnie tych samych zestawach dla każdego modelu

#data_folder = "/content/drive/MyDrive/MLDD - ADMET/data_splits"

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from tdc.single_pred import ADME
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_auc_score, accuracy_score, f1_score

In [5]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, MolSurf, AllChem
from sklearn.preprocessing import StandardScaler

class ADMETDescriptorFeaturizer:
    def __init__(self, y_column, smiles_col='Drug'):
        self.y_column = y_column
        self.smiles_col = smiles_col
        # Lista nazw deskryptorów, które wyliczamy
        self.feature_names = [
            'MW', 'LogP', 'HBD', 'HBA', 'TPSA',
            'RotatableBonds', 'AromaticRings', 'HeavyAtoms',
            'MolMR', 'FractionCSP3'
        ]

        self.scaler = StandardScaler()

    def __call__(self, df, fit_scaler=True):
        features = []
        labels = []

        for i, row in df.iterrows():
            smiles = row[self.smiles_col]
            y = row[self.y_column]
            mol = Chem.MolFromSmiles(smiles)

            if mol:
                desc_vector = [
                    # 1-4: Reguła Lipinskiego
                    Descriptors.MolWt(mol),          # Masa cząsteczkowa
                    Descriptors.MolLogP(mol),        # Lipofilowość
                    Lipinski.NumHDonors(mol),        # Donory H
                    Lipinski.NumHAcceptors(mol),      # Akceptor H

                    # 5-6: Reguły Vebera
                    Descriptors.TPSA(mol),           # Powierzchnia polarna
                    Lipinski.NumRotatableBonds(mol), # Elastyczność

                    # 7-8: Model ESOL / Rozmiar
                    Lipinski.NumAromaticRings(mol),  # Pierścienie aromatyczne
                    Descriptors.HeavyAtomCount(mol), # Liczba atomów ciężkich

                    # 9-10: Polaryzowalność i Trójwymiarowość
                    Descriptors.MolMR(mol),          # Refrakcja molowa
                    Lipinski.FractionCSP3(mol)       # Stopień nasycenia (3D)
                ]
                features.append(desc_vector)
                labels.append(y)
            else:
                print(f"Błąd w SMILES na indeksie {i}: {smiles}")

        features_array = np.array(features)

        # Ważne: Deskryptory mają różne skale (MW to setki, HBD to jednostki)
        # Wykonujemy normalizację (Z-score scaling)
        if fit_scaler:
            # Uczymy skaler na danych treningowych i transformujemy je
            normalized_features = self.scaler.fit_transform(features_array)
        else:
            # Używamy wyuczonego skalera na danych testowych (unikamy data leakage)
            normalized_features = self.scaler.transform(features_array)

        return normalized_features, np.array(labels)

In [6]:
import pickle

def load_split_pickle(dataset_name):
    filepath = f"{data_folder}/{dataset_name}_split.pkl"

    with open(filepath, "rb") as f:
        split = pickle.load(f)

    return split["train"], split["test"]

In [7]:
def print_metrics(metrics, task='classification', weight_loss_func_name=None):
    print(f"\n{'='*40}")
    if weight_loss_func_name: # Check if func_name is provided and not None
        print(f"  Loss Weighting: {weight_loss_func_name}")
        print(f"{'='*40}")
    if task == 'classification':
        print(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}")
        print(f"  F1       : {metrics['test_metrics']['f1']:.4f}")
        print(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}")
    else:
        print(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}")
        print(f"  MAE      : {metrics['test_metrics']['mae']:.4f}")
        print(f"  R²       : {metrics['test_metrics']['r2']:.4f}")
    print(f"{'='*40}\n")


def save_metrics(metrics, dataset_name, filepath, task='classification', weight_loss_func_name=None, endpoint_group_name=None):
    with open(filepath, 'a') as f:
        f.write(f"\n{'='*40}\n")
        f.write(f"Endpoint    : {dataset_name}\n")
        if endpoint_group_name:
            f.write(f"Tasks       : {endpoint_group_name}\n")
        if weight_loss_func_name:
            f.write(f"Loss Weighting: {weight_loss_func_name}\n")
        f.write(f"{'='*40}\n")
        if task == 'classification':
            f.write(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}\n")
            f.write(f"  F1       : {metrics['test_metrics']['f1']:.4f}\n")
            f.write(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}\n")
        else:
            f.write(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}\n")
            f.write(f"  MAE      : {metrics['test_metrics']['mae']:.4f}\n")
            f.write(f"  R²       : {metrics['test_metrics']['r2']:.4f}\n")
        f.write(f"{'='*40}\n")

In [8]:
import numpy as np
import pandas as pd

def prepare_mtl_data_final(df_list, task_names, featurizer):
    # 1. Zebranie wszystkich unikalnych struktur
    all_drugs = set()
    for df in df_list:
        valid = df['Drug'].dropna().astype(str).unique()
        all_drugs.update(valid)

    # 2. Walidacja cząsteczek przez RDKit przed stworzeniem master_list
    # To zapobiega rozbieżnościom rozmiarów X i y
    safe_master_list = []
    for drug in sorted(list(all_drugs)):
        mol = Chem.MolFromSmiles(drug)
        if mol: safe_master_list.append(drug)

    drug_to_idx = {drug: i for i, drug in enumerate(safe_master_list)}
    n_samples = len(safe_master_list)

    # 3. Generowanie X (tylko dla poprawnych cząsteczek)
    df_temp = pd.DataFrame({'Drug': safe_master_list})
    X_features, _ = featurizer(df_temp)

    # Sprawdzenie czy X zawiera NaN (zabezpieczenie przed błędami featurizera)
    if np.isnan(X_features).any():
        X_features = np.nan_to_num(X_features)

    # 4. Mapowanie etykiet y (z zachowaniem NaN tylko w etykietach!)
    y_dict = {}
    for df, task in zip(df_list, task_names):
        y_vec = np.full((n_samples, 1), np.nan, dtype=np.float32)
        # Tworzymy mapę {Drug: Wynik}
        mapping = dict(zip(df['Drug'].astype(str), df['Y']))

        for drug, val in mapping.items():
            if drug in drug_to_idx and not pd.isna(val):
                y_vec[drug_to_idx[drug]] = val
        y_dict[task] = y_vec

    return X_features, y_dict

In [9]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [10]:
def train_hybrid_mtl_rf(X_train, y_train_dict, X_test, y_test_dict, reg_tasks, class_tasks):
    all_tasks = reg_tasks + class_tasks

    #tworzenie wspólnej macierzy
    Y_train_raw = np.hstack([y_train_dict[task] for task in all_tasks])
    Y_test_raw = np.hstack([y_test_dict[task] for task in all_tasks])

    Y_train_imputed = Y_train_raw.copy()
    Y_test_imputed = Y_test_raw.copy()

    print(">> Krok 1: Imputacja brakujących wartości (Pseudo-labeling)...")
    # Aby pozbyć się NaN, trenujemy szybkie modele bazowe.
    #Dla zadań regresyjnych używamy regresora, a dla klasyfikacyjnych klasyfikatora, wyciągając z niego prawdopodobieństwa, aby zachować ciągłość danych.
    for i, task in enumerate(all_tasks):
        known_train_idx = ~np.isnan(Y_train_raw[:, i])
        missing_train_idx = np.isnan(Y_train_raw[:, i])
        missing_test_idx = np.isnan(Y_test_raw[:, i])

        if task in reg_tasks:
            model_single = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
            model_single.fit(X_train[known_train_idx], Y_train_raw[known_train_idx, i])
            if np.any(missing_train_idx):
                Y_train_imputed[missing_train_idx, i] = model_single.predict(X_train[missing_train_idx])
            if np.any(missing_test_idx):
                Y_test_imputed[missing_test_idx, i] = model_single.predict(X_test[missing_test_idx])

        elif task in class_tasks:
            model_single = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1)
            model_single.fit(X_train[known_train_idx], Y_train_raw[known_train_idx, i].astype(int))
            if np.any(missing_train_idx):
                Y_train_imputed[missing_train_idx, i] = model_single.predict_proba(X_train[missing_train_idx])[:, 1]
            if np.any(missing_test_idx):
                Y_test_imputed[missing_test_idx, i] = model_single.predict_proba(X_test[missing_test_idx])[:, 1]

    print(">> Krok 2: Uruchamianie treningu wspólnego modelu Multi-Task Random Forest...")
    mtl_model = RandomForestRegressor(n_estimators=1000, max_depth=25, max_features="sqrt", random_state=42, n_jobs=-1)
    mtl_model.fit(X_train, Y_train_imputed)

    Y_pred = mtl_model.predict(X_test)

    # Budowanie słownika dopasowanego do funkcji print/save
    metrics_summary = {}
    for i, task in enumerate(all_tasks):
        known_test_idx = ~np.isnan(Y_test_raw[:, i])
        true_y = Y_test_raw[known_test_idx, i]
        pred_y = Y_pred[known_test_idx, i]

        if task in reg_tasks:
            metrics_summary[task] = {
                "task_type": "regression",  # Flaga pomocnicza dla pętli
                "test_metrics": {
                    "rmse": np.sqrt(mean_squared_error(true_y, pred_y)),
                    "mae": mean_absolute_error(true_y, pred_y),
                    "r2": r2_score(true_y, pred_y)
                }
            }
        elif task in class_tasks:
            pred_classes = (pred_y >= 0.5).astype(int)
            true_classes = true_y.astype(int)
            metrics_summary[task] = {
                "task_type": "classification",  # Flaga pomocnicza dla pętli
                "test_metrics": {
                    "accuracy": accuracy_score(true_y, pred_classes),
                    "f1": f1_score(true_y, pred_classes, zero_division=0),
                    "auroc": roc_auc_score(true_y, pred_y)
                }
            }

    return mtl_model, metrics_summary

In [11]:
# import os

# # 1. Definiowanie konfiguracji i ścieżek
# reg_tasks = ['Caco2_Wang', 'Lipophilicity_AstraZeneca']
# class_tasks = ['HIA_Hou']

# all_tasks = reg_tasks + class_tasks
# filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_absorpcja_fingerprints.txt"

# # 2. Inicjalizacja featurizera deskryptorów
# featurizer = ADMETDescriptorFeaturizer(y_column='Y')

# print(">> Ładowanie danych z plików pickle...")
# train_caco2, test_caco2 = load_split_pickle('Caco2_Wang')
# train_lipo, test_lipo = load_split_pickle('Lipophilicity_AstraZeneca')
# train_hia, test_hia = load_split_pickle('HIA_Hou')

# df_train_list = [train_caco2, train_lipo, train_hia]
# df_test_list = [test_caco2, test_lipo, test_hia]

# print(">> Przygotowywanie zintegrowanych zbiorów danych MTL...")

# # --- TUTAJ JEST POPRAWKA SCHEMATU ---
# # Sprytne lambdy: .assign(Y=0) bezpiecznie "dokleja" brakującą kolumnę Y,
# # dzięki czemu featurizer się nie krztusi, a my kontrolujemy fit_scaler.

# train_featurizer_wrapper = lambda df: featurizer(df.assign(Y=0), fit_scaler=True)
# test_featurizer_wrapper  = lambda df: featurizer(df.assign(Y=0), fit_scaler=False)

# # Przekazujemy odpowiednie wrappery do funkcji
# X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, train_featurizer_wrapper)
# X_test_mtl, y_test_dict   = prepare_mtl_data_final(df_test_list, all_tasks, test_featurizer_wrapper)
# # -------------------------------------

# print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
# mtl_model, metrics = train_hybrid_mtl_rf(
#     X_train_mtl, y_train_dict,
#     X_test_mtl, y_test_dict,
#     reg_tasks, class_tasks
# )

# # 3. Zabezpieczenie katalogu docelowego
# dirname = os.path.dirname(filepath)
# if dirname and not os.path.exists(dirname):
#     os.makedirs(dirname)

# # 4. Ewaluacja i zapis Twoimi funkcjami
# print("\n>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<")
# for task_name, task_data in metrics.items():
#     task_type = task_data["task_type"]
#     print(f"Endpoint: {task_name} ({task_type.upper()})")
#     print_metrics(task_data, task=task_type)
#     save_metrics(
#         metrics=task_data,
#         dataset_name=task_name,
#         filepath=filepath,
#         task=task_type,
#         endpoint_group_name="MTL_Absorpcja_Mix"
#     )

# print(f"[SUCCESS] Proces zakończony bezbłędnie. Wyniki dopisano do: {filepath}")

# Test 1: hERG (Wang) + Lipophilicity (AstraZeneca)

In [12]:
import os

# 1. Definiowanie konfiguracji i ścieżek
reg_tasks = ['Lipophilicity_AstraZeneca']
class_tasks = ['hERG']

all_tasks = reg_tasks + class_tasks
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_descriptors_rf.txt"

# 2. Inicjalizacja featurizera deskryptorów
featurizer = ADMETDescriptorFeaturizer(y_column='Y')

print(">> Ładowanie danych z plików pickle...")
train_lipo, test_lipo = load_split_pickle('Lipophilicity_AstraZeneca')
train_herg, test_herg = load_split_pickle('hERG')

df_train_list = [train_lipo, train_herg]
df_test_list = [test_lipo, test_herg]

print(">> Przygotowywanie zintegrowanych zbiorów danych MTL...")

# --- TUTAJ JEST POPRAWKA SCHEMATU ---
# Sprytne lambdy: .assign(Y=0) bezpiecznie "dokleja" brakującą kolumnę Y,
# dzięki czemu featurizer się nie krztusi, a my kontrolujemy fit_scaler.

train_featurizer_wrapper = lambda df: featurizer(df.assign(Y=0), fit_scaler=True)
test_featurizer_wrapper  = lambda df: featurizer(df.assign(Y=0), fit_scaler=False)

# Przekazujemy odpowiednie wrappery do funkcji
X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, train_featurizer_wrapper)
X_test_mtl, y_test_dict   = prepare_mtl_data_final(df_test_list, all_tasks, test_featurizer_wrapper)
# -------------------------------------

print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
mtl_model, metrics = train_hybrid_mtl_rf(
    X_train_mtl, y_train_dict,
    X_test_mtl, y_test_dict,
    reg_tasks, class_tasks
)

# 3. Zabezpieczenie katalogu docelowego
dirname = os.path.dirname(filepath)
if dirname and not os.path.exists(dirname):
    os.makedirs(dirname)

# 4. Ewaluacja i zapis Twoimi funkcjami
print("\n>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<")
for task_name, task_data in metrics.items():
    task_type = task_data["task_type"]
    print(f"Endpoint: {task_name} ({task_type.upper()})")
    print_metrics(task_data, task=task_type)
    save_metrics(
        metrics=task_data,
        dataset_name=task_name,
        filepath=filepath,
        task=task_type,
        endpoint_group_name="herg + Lipophilicity"
    )

print(f"[SUCCESS] Proces zakończony bezbłędnie. Wyniki dopisano do: {filepath}")

>> Ładowanie danych z plików pickle...
>> Przygotowywanie zintegrowanych zbiorów danych MTL...


[14:16:48] WARNING: not removing hydrogen atom without neighbors
[14:16:49] WARNING: not removing hydrogen atom without neighbors
[14:16:52] WARNING: not removing hydrogen atom without neighbors
[14:16:56] WARNING: not removing hydrogen atom without neighbors


>> Uruchamianie hybrydowego treningu Multi-Task Random Forest...
>> Krok 1: Imputacja brakujących wartości (Pseudo-labeling)...
>> Krok 2: Uruchamianie treningu wspólnego modelu Multi-Task Random Forest...

>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<
Endpoint: Lipophilicity_AstraZeneca (REGRESSION)

  RMSE     : 0.8587
  MAE      : 0.6433
  R²       : 0.5010

Endpoint: hERG (CLASSIFICATION)

  Accuracy : 0.8473
  F1       : 0.9029
  AUROC    : 0.8217

[SUCCESS] Proces zakończony bezbłędnie. Wyniki dopisano do: /content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_descriptors_rf.txt


# Test2: hERG (Wang) + Lipophilicity (AstraZeneca) + Solubility (AqSolDB)

Sprawdzamy czy dołożenie kolejnego powiązanego zbioru poprawi wynik.

In [13]:
import os

# 1. Definiowanie konfiguracji i ścieżek
reg_tasks = ['Lipophilicity_AstraZeneca', 'Solubility_AqSolDB']
class_tasks = ['hERG']

all_tasks = reg_tasks + class_tasks
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_descriptors_rf.txt"

# 2. Inicjalizacja featurizera deskryptorów
featurizer = ADMETDescriptorFeaturizer(y_column='Y')

print(">> Ładowanie danych z plików pickle...")
train_herg, test_herg = load_split_pickle('hERG')
train_lipo, test_lipo = load_split_pickle('Lipophilicity_AstraZeneca')
train_sol, test_sol = load_split_pickle('Solubility_AqSolDB')

df_train_list = [train_lipo, train_sol, train_herg]
df_test_list = [test_lipo, test_sol, test_herg]

print(">> Przygotowywanie zintegrowanych zbiorów danych MTL...")

# --- TUTAJ JEST POPRAWKA SCHEMATU ---
# Sprytne lambdy: .assign(Y=0) bezpiecznie "dokleja" brakującą kolumnę Y,
# dzięki czemu featurizer się nie krztusi, a my kontrolujemy fit_scaler.

train_featurizer_wrapper = lambda df: featurizer(df.assign(Y=0), fit_scaler=True)
test_featurizer_wrapper  = lambda df: featurizer(df.assign(Y=0), fit_scaler=False)

# Przekazujemy odpowiednie wrappery do funkcji
X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, train_featurizer_wrapper)
X_test_mtl, y_test_dict   = prepare_mtl_data_final(df_test_list, all_tasks, test_featurizer_wrapper)
# -------------------------------------

print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
mtl_model, metrics = train_hybrid_mtl_rf(
    X_train_mtl, y_train_dict,
    X_test_mtl, y_test_dict,
    reg_tasks, class_tasks
)

# 3. Zabezpieczenie katalogu docelowego
dirname = os.path.dirname(filepath)
if dirname and not os.path.exists(dirname):
    os.makedirs(dirname)

# 4. Ewaluacja i zapis Twoimi funkcjami
print("\n>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<")
for task_name, task_data in metrics.items():
    task_type = task_data["task_type"]
    print(f"Endpoint: {task_name} ({task_type.upper()})")
    print_metrics(task_data, task=task_type)
    save_metrics(
        metrics=task_data,
        dataset_name=task_name,
        filepath=filepath,
        task=task_type,
        endpoint_group_name="herg + Lipophilicity + Solubility"
    )

print(f"[SUCCESS] Proces zakończony bezbłędnie. Wyniki dopisano do: {filepath}")

>> Ładowanie danych z plików pickle...
>> Przygotowywanie zintegrowanych zbiorów danych MTL...


[14:17:15] WARNING: not removing hydrogen atom without neighbors
[14:17:15] WARNING: not removing hydrogen atom without neighbors
[14:17:15] WARNING: not removing hydrogen atom without neighbors
[14:17:15] WARNING: not removing hydrogen atom without neighbors
[14:17:15] WARNING: not removing hydrogen atom without neighbors
[14:17:15] Explicit valence for atom # 5 N, 4, is greater than permitted
[14:17:15] WARNING: not removing hydrogen atom without neighbors
[14:17:15] WARNING: not removing hydrogen atom without neighbors
[14:17:15] WARNING: not removing hydrogen atom without neighbors
[14:17:16] WARNING: not removing hydrogen atom without neighbors
[14:17:16] WARNING: not removing hydrogen atom without neighbors
[14:17:16] WARNING: not removing hydrogen atom without neighbors
[14:17:16] WARNING: not removing hydrogen atom without neighbors
[14:17:16] WARNING: not removing hydrogen atom without neighbors
[14:17:16] WARNING: not removing hydrogen atom without neighbors
[14:17:16] WARNIN

>> Uruchamianie hybrydowego treningu Multi-Task Random Forest...
>> Krok 1: Imputacja brakujących wartości (Pseudo-labeling)...
>> Krok 2: Uruchamianie treningu wspólnego modelu Multi-Task Random Forest...

>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<
Endpoint: Lipophilicity_AstraZeneca (REGRESSION)

  RMSE     : 0.8683
  MAE      : 0.6523
  R²       : 0.4897

Endpoint: Solubility_AqSolDB (REGRESSION)

  RMSE     : 1.1102
  MAE      : 0.7576
  R²       : 0.7729

Endpoint: hERG (CLASSIFICATION)

  Accuracy : 0.8397
  F1       : 0.8976
  AUROC    : 0.8292

[SUCCESS] Proces zakończony bezbłędnie. Wyniki dopisano do: /content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_descriptors_rf.txt


# Test3: hERG (Wang) + Lipophilicity (AstraZeneca) + Solubility (AqSolDB) + VDss (Lombardo)

Sprawdzamy czy dołożenie kolejnego powiązanego zbioru poprawi wynik. Czy w jakimś momencie wyniki przestają się poprawiać?

In [14]:
import os

# 1. Definiowanie konfiguracji i ścieżek
reg_tasks = [ 'Lipophilicity_AstraZeneca', 'Solubility_AqSolDB', 'VDss_Lombardo']
class_tasks = ['hERG']

all_tasks = reg_tasks + class_tasks
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_descriptors_rf.txt"

# 2. Inicjalizacja featurizera deskryptorów
featurizer = ADMETDescriptorFeaturizer(y_column='Y')

print(">> Ładowanie danych z plików pickle...")
train_herg, test_herg = load_split_pickle('hERG')
train_lipo, test_lipo = load_split_pickle('Lipophilicity_AstraZeneca')
train_sol, test_sol = load_split_pickle('Solubility_AqSolDB')
train_vdss, test_vdss = load_split_pickle('VDss_Lombardo')

df_train_list = [train_lipo, train_sol, train_vdss, train_herg]
df_test_list = [test_lipo, test_sol, test_vdss, test_herg]


print(">> Przygotowywanie zintegrowanych zbiorów danych MTL...")

# --- TUTAJ JEST POPRAWKA SCHEMATU ---
# Sprytne lambdy: .assign(Y=0) bezpiecznie "dokleja" brakującą kolumnę Y,
# dzięki czemu featurizer się nie krztusi, a my kontrolujemy fit_scaler.

train_featurizer_wrapper = lambda df: featurizer(df.assign(Y=0), fit_scaler=True)
test_featurizer_wrapper  = lambda df: featurizer(df.assign(Y=0), fit_scaler=False)

# Przekazujemy odpowiednie wrappery do funkcji
X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, train_featurizer_wrapper)
X_test_mtl, y_test_dict   = prepare_mtl_data_final(df_test_list, all_tasks, test_featurizer_wrapper)
# -------------------------------------

print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
mtl_model, metrics = train_hybrid_mtl_rf(
    X_train_mtl, y_train_dict,
    X_test_mtl, y_test_dict,
    reg_tasks, class_tasks
)

# 3. Zabezpieczenie katalogu docelowego
dirname = os.path.dirname(filepath)
if dirname and not os.path.exists(dirname):
    os.makedirs(dirname)

# 4. Ewaluacja i zapis Twoimi funkcjami
print("\n>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<")
for task_name, task_data in metrics.items():
    task_type = task_data["task_type"]
    print(f"Endpoint: {task_name} ({task_type.upper()})")
    print_metrics(task_data, task=task_type)
    save_metrics(
        metrics=task_data,
        dataset_name=task_name,
        filepath=filepath,
        task=task_type,
        endpoint_group_name="herg + Lipophilicity + Solubility + VDss"
    )

print(f"[SUCCESS] Proces zakończony bezbłędnie. Wyniki dopisano do: {filepath}")

>> Ładowanie danych z plików pickle...
>> Przygotowywanie zintegrowanych zbiorów danych MTL...


[14:18:20] WARNING: not removing hydrogen atom without neighbors
[14:18:20] WARNING: not removing hydrogen atom without neighbors
[14:18:20] WARNING: not removing hydrogen atom without neighbors
[14:18:20] WARNING: not removing hydrogen atom without neighbors
[14:18:20] WARNING: not removing hydrogen atom without neighbors
[14:18:20] Explicit valence for atom # 5 N, 4, is greater than permitted
[14:18:20] WARNING: not removing hydrogen atom without neighbors
[14:18:20] WARNING: not removing hydrogen atom without neighbors
[14:18:21] WARNING: not removing hydrogen atom without neighbors
[14:18:21] WARNING: not removing hydrogen atom without neighbors
[14:18:21] WARNING: not removing hydrogen atom without neighbors
[14:18:21] WARNING: not removing hydrogen atom without neighbors
[14:18:21] WARNING: not removing hydrogen atom without neighbors
[14:18:21] WARNING: not removing hydrogen atom without neighbors
[14:18:21] WARNING: not removing hydrogen atom without neighbors
[14:18:21] WARNIN

>> Uruchamianie hybrydowego treningu Multi-Task Random Forest...
>> Krok 1: Imputacja brakujących wartości (Pseudo-labeling)...
>> Krok 2: Uruchamianie treningu wspólnego modelu Multi-Task Random Forest...

>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<
Endpoint: Lipophilicity_AstraZeneca (REGRESSION)

  RMSE     : 0.8680
  MAE      : 0.6576
  R²       : 0.4901

Endpoint: Solubility_AqSolDB (REGRESSION)

  RMSE     : 1.1216
  MAE      : 0.7673
  R²       : 0.7682

Endpoint: VDss_Lombardo (REGRESSION)

  RMSE     : 5.6664
  MAE      : 3.3597
  R²       : 0.3146

Endpoint: hERG (CLASSIFICATION)

  Accuracy : 0.8397
  F1       : 0.8976
  AUROC    : 0.8397

[SUCCESS] Proces zakończony bezbłędnie. Wyniki dopisano do: /content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_descriptors_rf.txt


# Test4: hERG (Wang) + VDss (Lombardo)

In [15]:
import os

# 1. Definiowanie konfiguracji i ścieżek
reg_tasks = ['VDss_Lombardo']
class_tasks = ['hERG']

all_tasks = reg_tasks + class_tasks
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_descriptors_rf.txt"

# 2. Inicjalizacja featurizera deskryptorów
featurizer = ADMETDescriptorFeaturizer(y_column='Y')

print(">> Ładowanie danych z plików pickle...")
train_herg, test_herg = load_split_pickle('hERG')
train_vdss, test_vdss = load_split_pickle('VDss_Lombardo')

df_train_list = [train_vdss, train_herg]
df_test_list = [test_vdss, test_herg]

print(">> Przygotowywanie zintegrowanych zbiorów danych MTL...")

# --- TUTAJ JEST POPRAWKA SCHEMATU ---
# Sprytne lambdy: .assign(Y=0) bezpiecznie "dokleja" brakującą kolumnę Y,
# dzięki czemu featurizer się nie krztusi, a my kontrolujemy fit_scaler.

train_featurizer_wrapper = lambda df: featurizer(df.assign(Y=0), fit_scaler=True)
test_featurizer_wrapper  = lambda df: featurizer(df.assign(Y=0), fit_scaler=False)

# Przekazujemy odpowiednie wrappery do funkcji
X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, train_featurizer_wrapper)
X_test_mtl, y_test_dict   = prepare_mtl_data_final(df_test_list, all_tasks, test_featurizer_wrapper)
# -------------------------------------

print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
mtl_model, metrics = train_hybrid_mtl_rf(
    X_train_mtl, y_train_dict,
    X_test_mtl, y_test_dict,
    reg_tasks, class_tasks
)

# 3. Zabezpieczenie katalogu docelowego
dirname = os.path.dirname(filepath)
if dirname and not os.path.exists(dirname):
    os.makedirs(dirname)

# 4. Ewaluacja i zapis Twoimi funkcjami
print("\n>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<")
for task_name, task_data in metrics.items():
    task_type = task_data["task_type"]
    print(f"Endpoint: {task_name} ({task_type.upper()})")
    print_metrics(task_data, task=task_type)
    save_metrics(
        metrics=task_data,
        dataset_name=task_name,
        filepath=filepath,
        task=task_type,
        endpoint_group_name="hERG + VDss"
    )

print(f"[SUCCESS] Proces zakończony bezbłędnie. Wyniki dopisano do: {filepath}")

>> Ładowanie danych z plików pickle...
>> Przygotowywanie zintegrowanych zbiorów danych MTL...


[14:19:29] WARNING: not removing hydrogen atom without neighbors
[14:19:29] WARNING: not removing hydrogen atom without neighbors
[14:19:30] WARNING: not removing hydrogen atom without neighbors
[14:19:31] WARNING: not removing hydrogen atom without neighbors


>> Uruchamianie hybrydowego treningu Multi-Task Random Forest...
>> Krok 1: Imputacja brakujących wartości (Pseudo-labeling)...
>> Krok 2: Uruchamianie treningu wspólnego modelu Multi-Task Random Forest...

>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<
Endpoint: VDss_Lombardo (REGRESSION)

  RMSE     : 5.6447
  MAE      : 3.2774
  R²       : 0.3198

Endpoint: hERG (CLASSIFICATION)

  Accuracy : 0.8244
  F1       : 0.8878
  AUROC    : 0.8242

[SUCCESS] Proces zakończony bezbłędnie. Wyniki dopisano do: /content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_descriptors_rf.txt


# Test5: hERG (Wang) + Solubility (AqSolDB)

In [16]:
import os

# 1. Definiowanie konfiguracji i ścieżek
reg_tasks = ['Solubility_AqSolDB']
class_tasks = ['hERG']

all_tasks = reg_tasks + class_tasks
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_descriptors_rf.txt"

# 2. Inicjalizacja featurizera deskryptorów
featurizer = ADMETDescriptorFeaturizer(y_column='Y')

print(">> Ładowanie danych z plików pickle...")
train_herg, test_herg = load_split_pickle('hERG')
train_sol, test_sol = load_split_pickle('Solubility_AqSolDB')

df_train_list = [train_sol, train_herg]
df_test_list = [test_sol, test_herg]

print(">> Przygotowywanie zintegrowanych zbiorów danych MTL...")

# --- TUTAJ JEST POPRAWKA SCHEMATU ---
# Sprytne lambdy: .assign(Y=0) bezpiecznie "dokleja" brakującą kolumnę Y,
# dzięki czemu featurizer się nie krztusi, a my kontrolujemy fit_scaler.

train_featurizer_wrapper = lambda df: featurizer(df.assign(Y=0), fit_scaler=True)
test_featurizer_wrapper  = lambda df: featurizer(df.assign(Y=0), fit_scaler=False)

# Przekazujemy odpowiednie wrappery do funkcji
X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, train_featurizer_wrapper)
X_test_mtl, y_test_dict   = prepare_mtl_data_final(df_test_list, all_tasks, test_featurizer_wrapper)
# -------------------------------------

print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
mtl_model, metrics = train_hybrid_mtl_rf(
    X_train_mtl, y_train_dict,
    X_test_mtl, y_test_dict,
    reg_tasks, class_tasks
)

# 3. Zabezpieczenie katalogu docelowego
dirname = os.path.dirname(filepath)
if dirname and not os.path.exists(dirname):
    os.makedirs(dirname)

# 4. Ewaluacja i zapis Twoimi funkcjami
print("\n>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<")
for task_name, task_data in metrics.items():
    task_type = task_data["task_type"]
    print(f"Endpoint: {task_name} ({task_type.upper()})")
    print_metrics(task_data, task=task_type)
    save_metrics(
        metrics=task_data,
        dataset_name=task_name,
        filepath=filepath,
        task=task_type,
        endpoint_group_name="hERG + Solubility"
    )

print(f"[SUCCESS] Proces zakończony bezbłędnie. Wyniki dopisano do: {filepath}")

>> Ładowanie danych z plików pickle...
>> Przygotowywanie zintegrowanych zbiorów danych MTL...


[14:19:39] WARNING: not removing hydrogen atom without neighbors
[14:19:39] WARNING: not removing hydrogen atom without neighbors
[14:19:39] WARNING: not removing hydrogen atom without neighbors
[14:19:39] WARNING: not removing hydrogen atom without neighbors
[14:19:39] WARNING: not removing hydrogen atom without neighbors
[14:19:39] Explicit valence for atom # 5 N, 4, is greater than permitted
[14:19:39] WARNING: not removing hydrogen atom without neighbors
[14:19:39] WARNING: not removing hydrogen atom without neighbors
[14:19:39] WARNING: not removing hydrogen atom without neighbors
[14:19:39] WARNING: not removing hydrogen atom without neighbors
[14:19:39] WARNING: not removing hydrogen atom without neighbors
[14:19:39] WARNING: not removing hydrogen atom without neighbors
[14:19:39] WARNING: not removing hydrogen atom without neighbors
[14:19:39] WARNING: not removing hydrogen atom without neighbors
[14:19:39] WARNING: not removing hydrogen atom without neighbors
[14:19:39] WARNIN

>> Uruchamianie hybrydowego treningu Multi-Task Random Forest...
>> Krok 1: Imputacja brakujących wartości (Pseudo-labeling)...
>> Krok 2: Uruchamianie treningu wspólnego modelu Multi-Task Random Forest...

>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<
Endpoint: Solubility_AqSolDB (REGRESSION)

  RMSE     : 1.1088
  MAE      : 0.7540
  R²       : 0.7735

Endpoint: hERG (CLASSIFICATION)

  Accuracy : 0.8321
  F1       : 0.8922
  AUROC    : 0.8227

[SUCCESS] Proces zakończony bezbłędnie. Wyniki dopisano do: /content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_descriptors_rf.txt


# Test6: Solubility (AqSolDB) + VDss (Lombardo)

Sprawdzamy czy zbiór niepowiązany pomoże zbiorowi CaCo-2.

In [17]:
import os

# 1. Definiowanie konfiguracji i ścieżek
reg_tasks = ['Solubility_AqSolDB', 'VDss_Lombardo']
class_tasks = []

all_tasks = reg_tasks + class_tasks
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_descriptors_rf.txt"

# 2. Inicjalizacja featurizera deskryptorów
featurizer = ADMETDescriptorFeaturizer(y_column='Y')

print(">> Ładowanie danych z plików pickle...")
train_vdss, test_vdss = load_split_pickle('VDss_Lombardo')
train_sol, test_sol = load_split_pickle('Solubility_AqSolDB')

df_train_list = [train_sol, train_vdss]
df_test_list = [test_sol, test_vdss]

print(">> Przygotowywanie zintegrowanych zbiorów danych MTL...")

# --- TUTAJ JEST POPRAWKA SCHEMATU ---
# Sprytne lambdy: .assign(Y=0) bezpiecznie "dokleja" brakującą kolumnę Y,
# dzięki czemu featurizer się nie krztusi, a my kontrolujemy fit_scaler.

train_featurizer_wrapper = lambda df: featurizer(df.assign(Y=0), fit_scaler=True)
test_featurizer_wrapper  = lambda df: featurizer(df.assign(Y=0), fit_scaler=False)

# Przekazujemy odpowiednie wrappery do funkcji
X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, train_featurizer_wrapper)
X_test_mtl, y_test_dict   = prepare_mtl_data_final(df_test_list, all_tasks, test_featurizer_wrapper)
# -------------------------------------

print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
mtl_model, metrics = train_hybrid_mtl_rf(
    X_train_mtl, y_train_dict,
    X_test_mtl, y_test_dict,
    reg_tasks, class_tasks
)

# 3. Zabezpieczenie katalogu docelowego
dirname = os.path.dirname(filepath)
if dirname and not os.path.exists(dirname):
    os.makedirs(dirname)

# 4. Ewaluacja i zapis Twoimi funkcjami
print("\n>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<")
for task_name, task_data in metrics.items():
    task_type = task_data["task_type"]
    print(f"Endpoint: {task_name} ({task_type.upper()})")
    print_metrics(task_data, task=task_type)
    save_metrics(
        metrics=task_data,
        dataset_name=task_name,
        filepath=filepath,
        task=task_type,
        endpoint_group_name="Solubility + VDss"
    )

print(f"[SUCCESS] Proces zakończony bezbłędnie. Wyniki dopisano do: {filepath}")

>> Ładowanie danych z plików pickle...
>> Przygotowywanie zintegrowanych zbiorów danych MTL...


[14:20:19] WARNING: not removing hydrogen atom without neighbors
[14:20:19] WARNING: not removing hydrogen atom without neighbors
[14:20:19] WARNING: not removing hydrogen atom without neighbors
[14:20:19] WARNING: not removing hydrogen atom without neighbors
[14:20:19] WARNING: not removing hydrogen atom without neighbors
[14:20:19] Explicit valence for atom # 5 N, 4, is greater than permitted
[14:20:20] WARNING: not removing hydrogen atom without neighbors
[14:20:20] WARNING: not removing hydrogen atom without neighbors
[14:20:20] WARNING: not removing hydrogen atom without neighbors
[14:20:20] WARNING: not removing hydrogen atom without neighbors
[14:20:20] WARNING: not removing hydrogen atom without neighbors
[14:20:20] WARNING: not removing hydrogen atom without neighbors
[14:20:20] WARNING: not removing hydrogen atom without neighbors
[14:20:20] WARNING: not removing hydrogen atom without neighbors
[14:20:20] WARNING: not removing hydrogen atom without neighbors
[14:20:20] WARNIN

>> Uruchamianie hybrydowego treningu Multi-Task Random Forest...
>> Krok 1: Imputacja brakujących wartości (Pseudo-labeling)...
>> Krok 2: Uruchamianie treningu wspólnego modelu Multi-Task Random Forest...

>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<
Endpoint: Solubility_AqSolDB (REGRESSION)

  RMSE     : 1.1132
  MAE      : 0.7632
  R²       : 0.7717

Endpoint: VDss_Lombardo (REGRESSION)

  RMSE     : 5.6158
  MAE      : 3.3862
  R²       : 0.3268

[SUCCESS] Proces zakończony bezbłędnie. Wyniki dopisano do: /content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_descriptors_rf.txt


# Test7: hERG (Wang) + AMES Mutagenicity

In [18]:
import os

# 1. Definiowanie konfiguracji i ścieżek
reg_tasks = []
class_tasks = ['hERG','AMES']

all_tasks = reg_tasks + class_tasks
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_descriptors_rf.txt"

# 2. Inicjalizacja featurizera deskryptorów
featurizer = ADMETDescriptorFeaturizer(y_column='Y')

print(">> Ładowanie danych z plików pickle...")
train_herg, test_herg = load_split_pickle('hERG')
train_ames, test_ames = load_split_pickle('AMES')

df_train_list = [train_herg, train_ames]
df_test_list = [test_herg, test_ames]

print(">> Przygotowywanie zintegrowanych zbiorów danych MTL...")

# --- TUTAJ JEST POPRAWKA SCHEMATU ---
# Sprytne lambdy: .assign(Y=0) bezpiecznie "dokleja" brakującą kolumnę Y,
# dzięki czemu featurizer się nie krztusi, a my kontrolujemy fit_scaler.

train_featurizer_wrapper = lambda df: featurizer(df.assign(Y=0), fit_scaler=True)
test_featurizer_wrapper  = lambda df: featurizer(df.assign(Y=0), fit_scaler=False)

# Przekazujemy odpowiednie wrappery do funkcji
X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, train_featurizer_wrapper)
X_test_mtl, y_test_dict   = prepare_mtl_data_final(df_test_list, all_tasks, test_featurizer_wrapper)
# -------------------------------------

print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
mtl_model, metrics = train_hybrid_mtl_rf(
    X_train_mtl, y_train_dict,
    X_test_mtl, y_test_dict,
    reg_tasks, class_tasks
)

# 3. Zabezpieczenie katalogu docelowego
dirname = os.path.dirname(filepath)
if dirname and not os.path.exists(dirname):
    os.makedirs(dirname)

# 4. Ewaluacja i zapis Twoimi funkcjami
print("\n>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<")
for task_name, task_data in metrics.items():
    task_type = task_data["task_type"]
    print(f"Endpoint: {task_name} ({task_type.upper()})")
    print_metrics(task_data, task=task_type)
    save_metrics(
        metrics=task_data,
        dataset_name=task_name,
        filepath=filepath,
        task=task_type,
        endpoint_group_name="hERG + AMES"
    )

print(f"[SUCCESS] Proces zakończony bezbłędnie. Wyniki dopisano do: {filepath}")

>> Ładowanie danych z plików pickle...
>> Przygotowywanie zintegrowanych zbiorów danych MTL...


[14:21:03] WARNING: not removing hydrogen atom without neighbors
[14:21:03] WARNING: not removing hydrogen atom without neighbors
[14:21:05] WARNING: not removing hydrogen atom without neighbors
[14:21:08] WARNING: not removing hydrogen atom without neighbors


>> Uruchamianie hybrydowego treningu Multi-Task Random Forest...
>> Krok 1: Imputacja brakujących wartości (Pseudo-labeling)...
>> Krok 2: Uruchamianie treningu wspólnego modelu Multi-Task Random Forest...

>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<
Endpoint: hERG (CLASSIFICATION)

  Accuracy : 0.8473
  F1       : 0.9029
  AUROC    : 0.8159

Endpoint: AMES (CLASSIFICATION)

  Accuracy : 0.7923
  F1       : 0.8108
  AUROC    : 0.8740

[SUCCESS] Proces zakończony bezbłędnie. Wyniki dopisano do: /content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_descriptors_rf.txt


# Test8: hERG (Wang) + Lipophilicity (AstraZeneca) + Solubility (AqSolDB) + VDss (Lombardo) + AMES Mutagenicity

In [19]:
import os

# 1. Definiowanie konfiguracji i ścieżek
reg_tasks = ['Lipophilicity_AstraZeneca', 'Solubility_AqSolDB', 'VDss_Lombardo']
class_tasks = ['hERG', 'AMES']

all_tasks = reg_tasks + class_tasks
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_descriptors_rf.txt"

# 2. Inicjalizacja featurizera deskryptorów
featurizer = ADMETDescriptorFeaturizer(y_column='Y')

print(">> Ładowanie danych z plików pickle...")
train_lipo, test_lipo = load_split_pickle('Lipophilicity_AstraZeneca')
train_sol, test_sol = load_split_pickle('Solubility_AqSolDB')
train_vdss, test_vdss = load_split_pickle('VDss_Lombardo')
train_herg, test_herg = load_split_pickle('hERG')
train_ames, test_ames = load_split_pickle('AMES')
df_train_list = [train_lipo, train_sol, train_vdss, train_herg, train_ames]
df_test_list = [test_lipo, test_sol, test_vdss, test_herg, test_ames]

print(">> Przygotowywanie zintegrowanych zbiorów danych MTL...")

# --- TUTAJ JEST POPRAWKA SCHEMATU ---
# Sprytne lambdy: .assign(Y=0) bezpiecznie "dokleja" brakującą kolumnę Y,
# dzięki czemu featurizer się nie krztusi, a my kontrolujemy fit_scaler.

train_featurizer_wrapper = lambda df: featurizer(df.assign(Y=0), fit_scaler=True)
test_featurizer_wrapper  = lambda df: featurizer(df.assign(Y=0), fit_scaler=False)

# Przekazujemy odpowiednie wrappery do funkcji
X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, train_featurizer_wrapper)
X_test_mtl, y_test_dict   = prepare_mtl_data_final(df_test_list, all_tasks, test_featurizer_wrapper)
# -------------------------------------

print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
mtl_model, metrics = train_hybrid_mtl_rf(
    X_train_mtl, y_train_dict,
    X_test_mtl, y_test_dict,
    reg_tasks, class_tasks
)

# 3. Zabezpieczenie katalogu docelowego
dirname = os.path.dirname(filepath)
if dirname and not os.path.exists(dirname):
    os.makedirs(dirname)

# 4. Ewaluacja i zapis Twoimi funkcjami
print("\n>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<")
for task_name, task_data in metrics.items():
    task_type = task_data["task_type"]
    print(f"Endpoint: {task_name} ({task_type.upper()})")
    print_metrics(task_data, task=task_type)
    save_metrics(
        metrics=task_data,
        dataset_name=task_name,
        filepath=filepath,
        task=task_type,
        endpoint_group_name="hERG (Wang) + Lipophilicity (AstraZeneca) + Solubility (AqSolDB) + VDss (Lombardo) + AMES Mutagenicity"
    )

print(f"[SUCCESS] Proces zakończony bezbłędnie. Wyniki dopisano do: {filepath}")

>> Ładowanie danych z plików pickle...
>> Przygotowywanie zintegrowanych zbiorów danych MTL...


[14:21:27] WARNING: not removing hydrogen atom without neighbors
[14:21:27] WARNING: not removing hydrogen atom without neighbors
[14:21:27] WARNING: not removing hydrogen atom without neighbors
[14:21:27] WARNING: not removing hydrogen atom without neighbors
[14:21:27] WARNING: not removing hydrogen atom without neighbors
[14:21:27] Explicit valence for atom # 5 N, 4, is greater than permitted
[14:21:27] WARNING: not removing hydrogen atom without neighbors
[14:21:27] WARNING: not removing hydrogen atom without neighbors
[14:21:27] WARNING: not removing hydrogen atom without neighbors
[14:21:27] WARNING: not removing hydrogen atom without neighbors
[14:21:28] WARNING: not removing hydrogen atom without neighbors
[14:21:28] WARNING: not removing hydrogen atom without neighbors
[14:21:28] WARNING: not removing hydrogen atom without neighbors
[14:21:28] WARNING: not removing hydrogen atom without neighbors
[14:21:28] WARNING: not removing hydrogen atom without neighbors
[14:21:28] WARNIN

>> Uruchamianie hybrydowego treningu Multi-Task Random Forest...
>> Krok 1: Imputacja brakujących wartości (Pseudo-labeling)...
>> Krok 2: Uruchamianie treningu wspólnego modelu Multi-Task Random Forest...

>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<
Endpoint: Lipophilicity_AstraZeneca (REGRESSION)

  RMSE     : 0.8707
  MAE      : 0.6596
  R²       : 0.4868

Endpoint: Solubility_AqSolDB (REGRESSION)

  RMSE     : 1.1224
  MAE      : 0.7650
  R²       : 0.7678

Endpoint: VDss_Lombardo (REGRESSION)

  RMSE     : 5.6500
  MAE      : 3.3237
  R²       : 0.3185

Endpoint: hERG (CLASSIFICATION)

  Accuracy : 0.8626
  F1       : 0.9118
  AUROC    : 0.8415

Endpoint: AMES (CLASSIFICATION)

  Accuracy : 0.7730
  F1       : 0.7945
  AUROC    : 0.8641

[SUCCESS] Proces zakończony bezbłędnie. Wyniki dopisano do: /content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_descriptors_rf.txt
